# Phase 8: Real-World Domain Adaptation (Fine-Tuning)

This notebook tackles the **Domain Shift** problem. Our model trained purely on synthetic, digitally generated PDFs (Phase 3). When evaluating on real-world mobile photographs, handwritten notes, or email screenshots, it fails because it has never seen that data distribution.

Here, we will:
1. Clone the IntegriDoc repository.
2. Load the `DocTamper` real-world dataset.
3. Load our pre-trained ResNet18 weights from Phase 3.
4. Fine-tune the model with a very low learning rate (`1e-5`) for a few epochs.
5. Save the adapted weights for deployment in our FastAPI UI.

In [ ]:
!pip install torch torchvision pillow pyyaml tqdm opencv-python

# Clone the repository so we have access to the codebase and configs
!git clone https://github.com/ivaaneoski/IntegriDoc.git
%cd IntegriDoc

# Ensure project structure
import sys
import os
sys.path.append(os.path.abspath('.'))

### 1. Setup the DocTamper Dataset
Since DocTamper is an academic dataset, you will need to request access and upload the zip here. For demonstration, we run our setup script to prepare the directory.

In [ ]:
!python scripts/download_doctamper.py --output_dir data/doctamper

### 2. Run the Fine-Tuning Pipeline
We use our `train_resnet.py` script but point it to the `finetune_resnet18.yaml` configuration, which loads our old weights and uses a smaller learning rate.

In [ ]:
!python scripts/train_resnet.py --config configs/finetune_resnet18.yaml

### 3. Download the Adapted Weights
Once training is complete, download the new weights and replace the ones in your local `results/runs/resnet18_baseline/best.pt` so the FastAPI server picks them up.

In [ ]:
import shutil
from google.colab import files

# Find the latest run directory
import glob
import os

runs = glob.glob('results/runs/resnet18_doctamper_finetune*')
latest_run = max(runs, key=os.path.getctime)
best_weights = os.path.join(latest_run, 'best.pt')

print(f"Downloading {best_weights}")
files.download(best_weights)